
# Camera CSV → ASCII PLY Converter (optional CRS transform + raw image folder filter)

This notebook converts camera-position CSV files (e.g. RealityCapture / SfM exports) to:

- **ASCII PLY** point clouds (vertices only, no edges)
- optional **TXT** sidecar files with: `image_name, x, y, z` (**default = True**)

It supports:

- a single **`.csv`**
- or a **folder** (recursive, all `.csv`)

## Default behavior
- **Use `x,y,z` as given** (no transform)
- Export **ASCII PLY**
- Export **TXT** sidecar (**enabled by default**)

## Optional CRS conversion
You can optionally set:
- **INPUT_CRS** (e.g. `EPSG:31370`, `EPSG:3812`, `EPSG:4979`)
- **OUTPUT_CRS** (e.g. `EPSG:3812`, `EPSG:31370`)

If both are set, coordinates are transformed with **pyproj**.

## Optional raw image directory filter
You can filter rows by the **imagePath parent folder** before conversion.

Example raw image folder:
`L:\Projects\2022-10_Project_Erkki\Data\2025-09_Krommebrug\1_RawData\20260220\DJI_202602200959_673_RightFacade`

If filtering is enabled, the output file name becomes:
`DJI_202602200959_673_RightFacade.ply`

## Belgian correction grids (optional / automatic if PROJ can use them)
Default grid folder (Windows):
`C:\Code\_Orbitv1.0.4\orbit\tools\CSV2PLY`

That folder may contain:
- `_BD72LB72_ETRS89LB08.gsb` (horizontal grid)
- `hBG18.geo` (vertical geoid grid)


In [1]:

# If needed (run once), install dependencies:
# !pip install pyproj pandas tqdm ipywidgets

import os
import warnings
from pathlib import Path, PureWindowsPath
from typing import Optional, List, Dict, Any, Tuple

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

from pyproj import CRS
from pyproj import datadir as pyproj_datadir
from pyproj.transformer import TransformerGroup

print("Imports OK")


Imports OK


#### Config

In [2]:

# =========================
# Config (all hard-written paths + declarations here)
# =========================

# Input can be:
# - a single .csv file
# - a folder (recursive processing of all .csv files)
INPUT_PATH = r""  # e.g. r"C:\Data\Cameras.csv" or r"C:\Data\ProjectFolder"

# Column names (leave as None for auto-detect)
NAME_COL = None        # e.g. "#name"
IMAGEPATH_COL = None   # e.g. "imagePath" (needed for raw folder filtering; auto-detected if None)
X_COL = None           # e.g. "x"
Y_COL = None           # e.g. "y"
Z_COL = None           # e.g. "z"

# Default behavior: keep xyz as given (no transform)
INPUT_CRS = None   # e.g. "EPSG:31370"
OUTPUT_CRS = None  # e.g. "EPSG:3812"

# Optional local grid folder with Belgian correction grids
GRID_DIR = r"C:\Code\_Orbitv1.0.4\orbit\tools\CSV2PLY"

# Optional filtering by raw image directory (imagePath parent folder)
ENABLE_RAW_IMAGE_DIR_FILTER = False
RAW_IMAGE_DIRS = [
    # Add one folder per line, e.g.
    # r"L:\Projects\2022-10_Project_Erkki\Data\2025-09_Krommebrug\1_RawData\20260220\DJI_202602200959_673_RightFacade",
    # r"L:\Projects\...\DJI_202602200959_674_LeftFacade",
]

# Sidecar TXT export: image_name, x, y, z
EXPORT_TXT = True
TXT_DELIMITER = "\t"  # "\\t" for tab, "," for comma

# Output number formatting
DECIMALS = 6

# Overwrite existing outputs
OVERWRITE = True


In [3]:

# =========================
# Core helpers
# =========================

def _configure_proj_grids(grid_dir: Optional[str]) -> bool:
    if not grid_dir:
        return False
    p = Path(grid_dir)
    if not p.exists():
        warnings.warn(f"Grid directory does not exist: {p}")
        return False
    try:
        pyproj_datadir.append_data_dir(str(p))
    except Exception:
        pass
    os.environ.setdefault("PROJ_NETWORK", "OFF")
    return True


def _collect_csv_files(input_path: Path) -> List[Path]:
    if not input_path.exists():
        raise FileNotFoundError(f"Input path does not exist: {input_path}")
    if input_path.is_file():
        if input_path.suffix.lower() != ".csv":
            raise ValueError(f"Unsupported file type: {input_path.suffix} (expected .csv)")
        return [input_path]
    return sorted([p for p in input_path.rglob("*.csv") if p.is_file()])


def _safe_relative(path: Path, root: Path) -> Path:
    try:
        return path.relative_to(root)
    except Exception:
        return Path(path.name)


def _read_csv_auto(csv_path: Path) -> "pd.DataFrame":
    if pd is None:
        raise ImportError("pandas is required to read CSV files.")
    try:
        return pd.read_csv(csv_path, sep=None, engine="python")
    except Exception:
        return pd.read_csv(csv_path)


def _pick_column(df: "pd.DataFrame", user_col: Optional[str], candidates: List[str], label: str, required: bool=True) -> Optional[str]:
    cols = list(df.columns)

    if user_col:
        if user_col in cols:
            return user_col
        lower_map = {str(c).strip().lower(): c for c in cols}
        key = str(user_col).strip().lower()
        if key in lower_map:
            return lower_map[key]
        raise KeyError(f"{label} column '{user_col}' not found. Available columns: {cols}")

    lower_map = {str(c).strip().lower(): c for c in cols}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]

    normalized = {}
    for c in cols:
        cn = str(c).strip().lower().replace("#", "").replace(" ", "").replace("_", "")
        normalized[cn] = c
    for c in candidates:
        cn = c.lower().replace("#", "").replace(" ", "").replace("_", "")
        if cn in normalized:
            return normalized[cn]

    if required:
        raise KeyError(f"Could not auto-detect {label} column. Available columns: {cols}")
    return None


def _detect_columns(
    df: "pd.DataFrame",
    name_col: Optional[str] = None,
    imagepath_col: Optional[str] = None,
    x_col: Optional[str] = None,
    y_col: Optional[str] = None,
    z_col: Optional[str] = None,
) -> Dict[str, Optional[str]]:
    return {
        "name": _pick_column(df, name_col, ["#name", "name", "imageName", "filename", "file", "image"], "name", required=True),
        "imagePath": _pick_column(df, imagepath_col, ["imagePath", "image_path", "path", "filepath", "filePath"], "imagePath", required=False),
        "x": _pick_column(df, x_col, ["x", "X", "pos_x", "camera_x"], "x", required=True),
        "y": _pick_column(df, y_col, ["y", "Y", "pos_y", "camera_y"], "y", required=True),
        "z": _pick_column(df, z_col, ["z", "Z", "pos_z", "camera_z"], "z", required=True),
    }


def _build_transformer(input_crs: Optional[str], output_crs: Optional[str], grid_dir: Optional[str] = None):
    if not input_crs or not output_crs:
        return None, None, None, None
    _configure_proj_grids(grid_dir)
    src = CRS.from_user_input(input_crs)
    dst = CRS.from_user_input(output_crs)
    if src == dst:
        return None, src, dst, None
    tg = TransformerGroup(src, dst, always_xy=True)
    if not tg.transformers:
        raise RuntimeError(f"No transformation available from {src} to {dst}")
    return tg.transformers[0], src, dst, tg


def _apply_transform_if_requested(
    df: "pd.DataFrame",
    cols: Dict[str, Optional[str]],
    input_crs: Optional[str],
    output_crs: Optional[str],
    grid_dir: Optional[str] = None,
) -> Tuple["pd.DataFrame", Dict[str, Any]]:
    out = df.copy()
    for c in [cols["x"], cols["y"], cols["z"]]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    n_before = len(out)
    out = out.dropna(subset=[cols["x"], cols["y"], cols["z"]]).reset_index(drop=True)
    dropped = n_before - len(out)

    transformer, src_crs_obj, dst_crs_obj, tg = _build_transformer(input_crs, output_crs, grid_dir)
    info = {
        "transform_applied": False,
        "input_crs": str(src_crs_obj) if src_crs_obj is not None else None,
        "output_crs": str(dst_crs_obj) if dst_crs_obj is not None else None,
        "best_transform_available": getattr(tg, "best_available", None) if tg is not None else None,
        "rows_dropped_non_numeric_xyz": dropped,
    }

    if transformer is None:
        out["_X_OUT"] = out[cols["x"]].astype(float)
        out["_Y_OUT"] = out[cols["y"]].astype(float)
        out["_Z_OUT"] = out[cols["z"]].astype(float)
        return out, info

    xs = out[cols["x"]].astype(float).to_numpy()
    ys = out[cols["y"]].astype(float).to_numpy()
    zs = out[cols["z"]].astype(float).to_numpy()

    try:
        res = transformer.transform(xs, ys, zs)
        if len(res) >= 3:
            x2, y2, z2 = res[0], res[1], res[2]
        else:
            x2, y2 = res[0], res[1]
            z2 = zs
    except Exception:
        x2, y2 = transformer.transform(xs, ys)
        z2 = zs

    out["_X_OUT"] = x2
    out["_Y_OUT"] = y2
    out["_Z_OUT"] = z2
    info["transform_applied"] = True
    return out, info


def _norm_win_dir(p: str) -> str:
    """Normalize a path string to a Windows-like comparable directory string."""
    s = str(p).strip().strip('"').strip("'")
    s = s.replace("/", "\\")
    while "\\\\" in s:
        s = s.replace("\\\\", "\\")
    return s.rstrip("\\").lower()


def _basename_from_dir(p: str) -> str:
    s = str(p).strip().strip('"').strip("'").replace("/", "\\").rstrip("\\")
    return PureWindowsPath(s).name


def _filter_df_by_raw_image_dirs(
    df: "pd.DataFrame",
    imagepath_col: Optional[str],
    raw_image_dirs: List[str],
) -> List[Tuple[str, "pd.DataFrame"]]:
    """
    Returns list of (raw_dir, filtered_df).
    Matching is done on parent(imagePath) == raw_dir (normalized Windows style, case-insensitive).
    """
    if not raw_image_dirs:
        return []

    if imagepath_col is None or imagepath_col not in df.columns:
        raise KeyError(
            "Filtering by raw image directory requires an imagePath column in the CSV. "
            "Set IMAGEPATH_COL manually if auto-detection fails."
        )

    tmp = df.copy()

    def _parent_norm(x):
        if pd.isna(x):
            return None
        sx = str(x).strip()
        if not sx:
            return None
        # Works well even on non-Windows Python when using PureWindowsPath
        try:
            parent = str(PureWindowsPath(sx).parent)
        except Exception:
            # fallback string split
            sx2 = sx.replace("/", "\\")
            parent = sx2.rsplit("\\", 1)[0] if "\\" in sx2 else ""
        return _norm_win_dir(parent)

    tmp["_image_parent_norm"] = tmp[imagepath_col].map(_parent_norm)

    groups = []
    for raw_dir in raw_image_dirs:
        key = _norm_win_dir(raw_dir)
        sub = tmp[tmp["_image_parent_norm"] == key].copy().drop(columns=["_image_parent_norm"])
        groups.append((raw_dir, sub))

    return groups


def _write_ascii_ply(
    ply_path: Path,
    export_df: "pd.DataFrame",
    decimals: int = 6,
    overwrite: bool = True,
    header_comment: Optional[str] = None,
) -> Path:
    ply_path.parent.mkdir(parents=True, exist_ok=True)
    if ply_path.exists() and not overwrite:
        return ply_path

    n = len(export_df)
    with ply_path.open("w", encoding="utf-8", newline="\n") as f:
        f.write("ply\n")
        f.write("format ascii 1.0\n")
        f.write("comment Generated from camera position CSV\n")
        if header_comment:
            for line in str(header_comment).splitlines():
                f.write(f"comment {line}\n")
        f.write(f"element vertex {n}\n")
        f.write("property float x\n")
        f.write("property float y\n")
        f.write("property float z\n")
        f.write("end_header\n")

        fmt = f"{{:.{decimals}f}}"
        for _, row in export_df.iterrows():
            f.write(
                f"{fmt.format(float(row['_X_OUT']))} "
                f"{fmt.format(float(row['_Y_OUT']))} "
                f"{fmt.format(float(row['_Z_OUT']))}\n"
            )
    return ply_path


def _write_txt_xyz(
    txt_path: Path,
    export_df: "pd.DataFrame",
    name_col: str,
    decimals: int = 6,
    delimiter: str = "\t",
    overwrite: bool = True,
) -> Path:
    txt_path.parent.mkdir(parents=True, exist_ok=True)
    if txt_path.exists() and not overwrite:
        return txt_path

    fmt = f"{{:.{decimals}f}}"
    with txt_path.open("w", encoding="utf-8", newline="\n") as f:
        f.write(delimiter.join(["image_name", "x", "y", "z"]) + "\n")
        for _, row in export_df.iterrows():
            f.write(
                delimiter.join([
                    str(row[name_col]),
                    fmt.format(float(row["_X_OUT"])),
                    fmt.format(float(row["_Y_OUT"])),
                    fmt.format(float(row["_Z_OUT"])),
                ]) + "\n"
            )
    return txt_path


def _export_one_subset(
    export_source_df: "pd.DataFrame",
    cols: Dict[str, Optional[str]],
    csv_file: Path,
    rel_root: Path,
    ply_dir: Path,
    txt_dir: Path,
    input_crs: Optional[str],
    output_crs: Optional[str],
    grid_dir: Optional[str],
    export_txt: bool,
    txt_delimiter: str,
    decimals: int,
    overwrite: bool,
    out_stem_override: Optional[str] = None,
    subset_label: Optional[str] = None,
) -> Dict[str, Any]:
    out_df, tinfo = _apply_transform_if_requested(
        df=export_source_df,
        cols=cols,
        input_crs=input_crs,
        output_crs=output_crs,
        grid_dir=grid_dir,
    )

    export_df = out_df[[cols["name"], "_X_OUT", "_Y_OUT", "_Z_OUT"]].copy()

    rel_csv = _safe_relative(csv_file, rel_root)
    rel_parent = rel_csv.parent

    stem = out_stem_override if out_stem_override else csv_file.stem
    out_ply = (ply_dir / rel_parent / f"{stem}.ply")
    out_txt = (txt_dir / rel_parent / f"{stem}.txt")

    header_comment_bits = [f"Source CSV: {csv_file.name}"]
    if subset_label:
        header_comment_bits.append(f"Subset: {subset_label}")
    if tinfo["transform_applied"]:
        header_comment_bits.append(f"Transform: {tinfo['input_crs']} -> {tinfo['output_crs']}")
    else:
        header_comment_bits.append("No transform (xyz kept as given)")
    header_comment = " | ".join(header_comment_bits)

    _write_ascii_ply(
        ply_path=out_ply,
        export_df=export_df,
        decimals=decimals,
        overwrite=overwrite,
        header_comment=header_comment,
    )

    txt_written = None
    txt_err = None
    if export_txt:
        try:
            _write_txt_xyz(
                txt_path=out_txt,
                export_df=export_df,
                name_col=cols["name"],
                decimals=decimals,
                delimiter=txt_delimiter,
                overwrite=overwrite,
            )
            txt_written = str(out_txt)
        except Exception as e:
            txt_err = str(e)

    status = "OK" if txt_err is None else f"PLY OK, TXT ERROR: {txt_err}"

    return {
        "source_csv": str(csv_file),
        "subset_label": subset_label,
        "n_rows_input": len(export_source_df),
        "n_points_output": len(export_df),
        "name_col": cols["name"],
        "imagePath_col": cols.get("imagePath"),
        "x_col": cols["x"],
        "y_col": cols["y"],
        "z_col": cols["z"],
        "transform_applied": tinfo["transform_applied"],
        "input_crs": tinfo["input_crs"],
        "output_crs": tinfo["output_crs"],
        "rows_dropped_non_numeric_xyz": tinfo["rows_dropped_non_numeric_xyz"],
        "best_transform_available": tinfo["best_transform_available"],
        "ply_path": str(out_ply),
        "txt_path": txt_written,
        "status": status,
    }


In [4]:

# =========================
# Main processing function
# =========================

def process_camera_csv_to_ply(
    input_path: str,
    name_col: Optional[str] = None,
    imagepath_col: Optional[str] = None,
    x_col: Optional[str] = None,
    y_col: Optional[str] = None,
    z_col: Optional[str] = None,
    input_crs: Optional[str] = None,
    output_crs: Optional[str] = None,
    grid_dir: Optional[str] = None,
    export_txt: bool = True,
    txt_delimiter: str = "\t",
    decimals: int = 6,
    overwrite: bool = True,
    enable_raw_image_dir_filter: bool = False,
    raw_image_dirs: Optional[List[str]] = None,
):
    if pd is None:
        raise ImportError("pandas is required. Install it with: pip install pandas")

    input_path = Path(input_path)
    root = input_path if input_path.is_dir() else input_path.parent
    ply_dir = root / "PLY"
    txt_dir = root / "TXT"
    files = _collect_csv_files(input_path)
    if not files:
        raise FileNotFoundError("No CSV files found.")

    raw_image_dirs = [r for r in (raw_image_dirs or []) if str(r).strip()]
    records = []
    rel_root = root

    for csv_file in tqdm(files, desc="Processing CSV files"):
        try:
            df = _read_csv_auto(csv_file)
        except Exception as e:
            records.append({
                "source_csv": str(csv_file),
                "subset_label": None,
                "n_rows_input": 0,
                "n_points_output": 0,
                "ply_path": None,
                "txt_path": None,
                "status": f"ERROR reading CSV: {e}",
            })
            continue

        try:
            cols = _detect_columns(
                df,
                name_col=name_col,
                imagepath_col=imagepath_col,
                x_col=x_col,
                y_col=y_col,
                z_col=z_col,
            )
        except Exception as e:
            records.append({
                "source_csv": str(csv_file),
                "subset_label": None,
                "n_rows_input": len(df),
                "n_points_output": 0,
                "ply_path": None,
                "txt_path": None,
                "status": f"ERROR detecting columns: {e}",
            })
            continue

        # Case A: no filtering -> one export per CSV
        if not enable_raw_image_dir_filter:
            try:
                rec = _export_one_subset(
                    export_source_df=df,
                    cols=cols,
                    csv_file=csv_file,
                    rel_root=rel_root,
                    ply_dir=ply_dir,
                    txt_dir=txt_dir,
                    input_crs=input_crs,
                    output_crs=output_crs,
                    grid_dir=grid_dir,
                    export_txt=export_txt,
                    txt_delimiter=txt_delimiter,
                    decimals=decimals,
                    overwrite=overwrite,
                    out_stem_override=None,
                    subset_label=None,
                )
                records.append(rec)
            except Exception as e:
                records.append({
                    "source_csv": str(csv_file),
                    "subset_label": None,
                    "n_rows_input": len(df),
                    "n_points_output": 0,
                    "ply_path": None,
                    "txt_path": None,
                    "status": f"ERROR exporting: {e}",
                })
            continue

        # Case B: filtering enabled -> one export per raw image dir (subset)
        try:
            groups = _filter_df_by_raw_image_dirs(df, cols.get("imagePath"), raw_image_dirs)
        except Exception as e:
            records.append({
                "source_csv": str(csv_file),
                "subset_label": None,
                "n_rows_input": len(df),
                "n_points_output": 0,
                "ply_path": None,
                "txt_path": None,
                "status": f"ERROR filtering by raw image dir: {e}",
            })
            continue

        for raw_dir, sub_df in groups:
            subset_name = _basename_from_dir(raw_dir)
            if len(sub_df) == 0:
                records.append({
                    "source_csv": str(csv_file),
                    "subset_label": subset_name,
                    "n_rows_input": 0,
                    "n_points_output": 0,
                    "ply_path": None,
                    "txt_path": None,
                    "status": "No rows matched raw image directory",
                })
                continue

            try:
                rec = _export_one_subset(
                    export_source_df=sub_df,
                    cols=cols,
                    csv_file=csv_file,
                    rel_root=rel_root,
                    ply_dir=ply_dir,
                    txt_dir=txt_dir,
                    input_crs=input_crs,
                    output_crs=output_crs,
                    grid_dir=grid_dir,
                    export_txt=export_txt,
                    txt_delimiter=txt_delimiter,
                    decimals=decimals,
                    overwrite=overwrite,
                    out_stem_override=subset_name,  # e.g. DJI_202602200959_673_RightFacade
                    subset_label=subset_name,
                )
                records.append(rec)
            except Exception as e:
                records.append({
                    "source_csv": str(csv_file),
                    "subset_label": subset_name,
                    "n_rows_input": len(sub_df),
                    "n_points_output": 0,
                    "ply_path": None,
                    "txt_path": None,
                    "status": f"ERROR exporting subset: {e}",
                })

    summary = pd.DataFrame(records) if len(records) > 0 else pd.DataFrame()
    return records, summary


In [5]:

# =========================
# CLI / Run cell (uses the Config values above)
# =========================

if not INPUT_PATH:
    print("Set INPUT_PATH in the Config cell above (CSV file or folder) and run again.")
else:
    records, summary = process_camera_csv_to_ply(
        input_path=INPUT_PATH,
        name_col=NAME_COL,
        imagepath_col=IMAGEPATH_COL,
        x_col=X_COL,
        y_col=Y_COL,
        z_col=Z_COL,
        input_crs=INPUT_CRS,
        output_crs=OUTPUT_CRS,
        grid_dir=GRID_DIR,
        export_txt=EXPORT_TXT,
        txt_delimiter=TXT_DELIMITER,
        decimals=DECIMALS,
        overwrite=OVERWRITE,
        enable_raw_image_dir_filter=ENABLE_RAW_IMAGE_DIR_FILTER,
        raw_image_dirs=RAW_IMAGE_DIRS,
    )

    print("\nDone.")
    display(summary)


Set INPUT_PATH in the Config cell above (CSV file or folder) and run again.


#### CLI

In [13]:

# =========================
# Optional mini UI (checkbox + multiline raw dir filter)
# =========================

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    path_w = widgets.Text(
        value=INPUT_PATH if INPUT_PATH else "",
        placeholder=r"C:\Data\Cameras.csv or C:\Data\ProjectFolder",
        description="Input:",
        layout=widgets.Layout(width="88%"),
    )

    in_crs_w = widgets.Text(
        value="" if INPUT_CRS is None else str(INPUT_CRS),
        placeholder="e.g. EPSG:31370",
        description="Input CRS:",
        layout=widgets.Layout(width="46%"),
    )

    out_crs_w = widgets.Text(
        value="" if OUTPUT_CRS is None else str(OUTPUT_CRS),
        placeholder="e.g. EPSG:3812",
        description="Output CRS:",
        layout=widgets.Layout(width="46%"),
    )

    filter_checkbox = widgets.Checkbox(
        value=ENABLE_RAW_IMAGE_DIR_FILTER,
        description="Filter by raw image directory"
    )

    filter_dirs_text = widgets.Textarea(
        value="\\n".join(RAW_IMAGE_DIRS) if RAW_IMAGE_DIRS else "",
        placeholder=(
            "One raw image directory per line\\n"
            r"L:\Projects\...\DJI_202602200959_673_RightFacade"
        ),
        description="Raw dirs:",
        layout=widgets.Layout(width="88%", height="120px"),
    )

    txt_w = widgets.Checkbox(value=EXPORT_TXT, description="Export TXT")
    overwrite_w = widgets.Checkbox(value=OVERWRITE, description="Overwrite")
    run_btn = widgets.Button(description="Process", button_style="success")
    out = widgets.Output()

    # Toggle multiline raw-dir box visibility
    def _toggle_filter_box(change=None):
        filter_dirs_text.layout.display = "" if filter_checkbox.value else "none"

    filter_checkbox.observe(_toggle_filter_box, names="value")
    _toggle_filter_box()

    def _on_run(_):
        with out:
            clear_output()
            try:
                in_crs = in_crs_w.value.strip() or None
                out_crs = out_crs_w.value.strip() or None
                raw_dirs = [ln.strip() for ln in filter_dirs_text.value.splitlines() if ln.strip()]

                _, summary = process_camera_csv_to_ply(
                    input_path=path_w.value,
                    name_col=NAME_COL,
                    imagepath_col=IMAGEPATH_COL,
                    x_col=X_COL,
                    y_col=Y_COL,
                    z_col=Z_COL,
                    input_crs=in_crs,
                    output_crs=out_crs,
                    grid_dir=GRID_DIR,
                    export_txt=txt_w.value,
                    txt_delimiter=TXT_DELIMITER,
                    decimals=DECIMALS,
                    overwrite=overwrite_w.value,
                    enable_raw_image_dir_filter=filter_checkbox.value,
                    raw_image_dirs=raw_dirs,
                )
                print("Done.")
                display(summary)

            except Exception as e:
                print("ERROR:", e)

    run_btn.on_click(_on_run)

    display(widgets.VBox([
        path_w,
        widgets.HBox([in_crs_w, out_crs_w]),
        filter_checkbox,
        filter_dirs_text,
        widgets.HBox([txt_w, overwrite_w, run_btn]),
        out
    ]))

except Exception as e:
    print("ipywidgets UI not available:", e)



In [12]:
# print file path of output PLY files for easy copy-pasting:
print(out_txt)


NameError: name 'out_txt' is not defined